# Gemma 4 vs Qwen3.5 — Full Benchmark
Base test (no FT) → QLoRA fine-tune → 7-model comparison


## 1. Environment & Model Download

In [ ]:
import json
import os
import random
import re
import shutil
import statistics
import subprocess
import sys
import time
from collections import Counter, defaultdict
from datetime import UTC, datetime
from pathlib import Path

import yaml
from huggingface_hub import hf_hub_download

from scripts.common import read_jsonl, write_jsonl

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'config' / 'generation.yaml').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

GGUF_DIR = REPO_ROOT / 'artifacts' / 'gguf'
CONFIG_DIR = REPO_ROOT / 'config'
LLAMA_SERVER = REPO_ROOT / 'tools' / 'llama.cpp' / 'build' / 'bin' / 'llama-server'
LLAMA_QUANTIZE = REPO_ROOT / 'tools' / 'llama.cpp' / 'build' / 'bin' / 'llama-quantize'
CONVERT_SCRIPT = REPO_ROOT / 'tools' / 'llama.cpp' / 'convert_hf_to_gguf.py'


def load_yaml(path: Path):
    with path.open(encoding='utf-8') as handle:
        return yaml.safe_load(handle)


situations = load_yaml(REPO_ROOT / 'config/situations.yaml')['situations']
personalities = load_yaml(REPO_ROOT / 'config/personalities.yaml')['personalities']
emotions = load_yaml(REPO_ROOT / 'config/emotions.yaml')['emotions']
social_situations = load_yaml(REPO_ROOT / 'config/social_situations.yaml')['social_situations']
group_situations = load_yaml(REPO_ROOT / 'config/group_situations.yaml')['group_situations']
trade_scenarios = load_yaml(REPO_ROOT / 'config/trade_scenarios.yaml')['trade_scenarios']
stress_sources = load_yaml(REPO_ROOT / 'config/stress_sources.yaml')['stress_sources']
deception_scenarios = load_yaml(REPO_ROOT / 'config/deception_scenarios.yaml')['deception_scenarios']
rumor_scenarios = load_yaml(REPO_ROOT / 'config/rumor_scenarios.yaml')['rumor_scenarios']
trauma_scenarios = load_yaml(REPO_ROOT / 'config/trauma_scenarios.yaml')['trauma_scenarios']
negotiation_scenarios = load_yaml(REPO_ROOT / 'config/negotiation_scenarios.yaml')['negotiation_scenarios']
culture_scenarios = load_yaml(REPO_ROOT / 'config/culture_scenarios.yaml')['culture_scenarios']
group_dissent_scenarios = load_yaml(REPO_ROOT / 'config/group_dissent_scenarios.yaml')['group_dissent_scenarios']

GEMMA4_E2B_BASE_ID = 'google/gemma-4-E2B'
GEMMA4_E4B_BASE_ID = 'google/gemma-4-E4B'
GEMMA4_E2B_IT_ID = 'google/gemma-4-E2B-it'
GEMMA4_E4B_IT_ID = 'google/gemma-4-E4B-it'

# Verified on Hugging Face on 2026-04-04: base model IDs exist and can be used for fine-tuning.
GEMMA4_E2B_IT_GGUF = GGUF_DIR / 'gemma-4-E2B-it-Q4_K_M.gguf'
GEMMA4_E4B_IT_GGUF = GGUF_DIR / 'gemma-4-E4B-it-Q4_K_M.gguf'

for gguf_path, hf_repo, hf_file in [
    (GEMMA4_E2B_IT_GGUF, 'unsloth/gemma-4-E2B-it-GGUF', 'gemma-4-E2B-it-Q4_K_M.gguf'),
    (GEMMA4_E4B_IT_GGUF, 'unsloth/gemma-4-E4B-it-GGUF', 'gemma-4-E4B-it-Q4_K_M.gguf'),
]:
    if not gguf_path.exists():
        print(f'Downloading {hf_file}...', flush=True)
        downloaded = Path(
            hf_hub_download(
                repo_id=hf_repo,
                filename=hf_file,
                local_dir=str(GGUF_DIR),
                local_dir_use_symlinks=False,
            )
        )
        if downloaded != gguf_path and downloaded.exists() and not gguf_path.exists():
            shutil.copy2(downloaded, gguf_path)
        print(f'  {gguf_path.name} ({gguf_path.stat().st_size / 1e6:.0f} MB)')
    else:
        print(f'{gguf_path.name} ({gguf_path.stat().st_size / 1e6:.0f} MB)')

QWEN_08B_FT = GGUF_DIR / 'worldsim-v31-qwen3.5-0.8b-q4_k_m.gguf'
QWEN_2B_FT = GGUF_DIR / 'worldsim-v31-qwen3.5-2b-q4_k_m.gguf'
QWEN_4B_FT = GGUF_DIR / 'worldsim-v31-qwen3.5-4b-q4_k_m.gguf'

print('\n=== Model Inventory ===')
for name, path in [
    ('Qwen3.5-0.8B FT', QWEN_08B_FT),
    ('Qwen3.5-2B FT', QWEN_2B_FT),
    ('Qwen3.5-4B FT', QWEN_4B_FT),
    ('Gemma4-E2B IT (base)', GEMMA4_E2B_IT_GGUF),
    ('Gemma4-E4B IT (base)', GEMMA4_E4B_IT_GGUF),
]:
    if path.exists():
        print(f'  {name:.<30} {path.stat().st_size / 1e6:.0f} MB')
    else:
        print(f'  {name:.<30} NOT FOUND')


## 2. Server Helpers & Test Prompts

In [ ]:
import urllib.request

SERVER_PORT = 8090
SERVER_URL = f'http://127.0.0.1:{SERVER_PORT}'


def start_server(model_path: Path, ctx_size: int = 2048, n_gpu_layers: int = 99):
    args = [
        str(LLAMA_SERVER), '-m', str(model_path),
        '--host', '127.0.0.1', '--port', str(SERVER_PORT),
        '-c', str(ctx_size), '-np', '1', '-ngl', str(n_gpu_layers),
        '-fa', 'on', '--jinja', '--no-webui',
        '--chat-template-kwargs', '{"enable_thinking": false}',
        '-ctk', 'q8_0', '-ctv', 'q8_0',
    ]
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for attempt in range(120):
        time.sleep(0.5)
        try:
            resp = urllib.request.urlopen(f'{SERVER_URL}/health', timeout=2)
            data = json.loads(resp.read())
            if data.get('status') == 'ok':
                return proc
        except Exception:
            pass
        if proc.poll() is not None:
            stderr = proc.stderr.read().decode(errors='ignore')[-300:]
            raise RuntimeError(f'Server died: {stderr}')
    proc.kill()
    raise RuntimeError('Server startup timeout')


def stop_server(proc) -> None:
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()
            proc.wait()


def strip_think(text: str) -> str:
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


def query_model(messages, response_format=None, max_tokens: int = 256, temperature: float = 0):
    payload = {
        'messages': messages,
        'max_tokens': max_tokens,
        'temperature': temperature,
        'stream': False,
    }
    if response_format:
        payload['response_format'] = response_format

    req = urllib.request.Request(
        f'{SERVER_URL}/v1/chat/completions',
        data=json.dumps(payload).encode(),
        headers={'Content-Type': 'application/json'},
    )
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = json.loads(resp.read())
        content = data['choices'][0]['message']['content']
        return strip_think(content)
    except Exception as exc:
        return f'ERROR: {exc}'

rng = random.Random(42)

SYS_L3 = 'You are WorldSim logic assistant. Output JSON only. Follow [TEMP], [STRESS], [WORLD] context. Respect enum values and numeric ranges exactly.'
SYS_L4 = '너는 WorldSim 서사 도우미다. [TEMP], [STRESS], [WORLD]와 지정된 register를 반영해 JSON으로만 답하라.'
SYS_L5 = '너는 WorldSim 신탁 해석 도우미다. JSON으로만 답하라.'


def pick(items, n=1):
    return rng.sample(items, min(n, len(items)))


def temp_block(t):
    return f"NS={t['ns']} HA={t['ha']} RD={t['rd']} P={t['p']} type={t['id']}"


def personality_keywords(p):
    return ', '.join(p.get('keywords', []))


def json_schema(name: str, required: list[str], properties: dict) -> dict:
    return {
        'type': 'json_schema',
        'json_schema': {
            'name': name,
            'strict': True,
            'schema': {
                'type': 'object',
                'additionalProperties': False,
                'required': required,
                'properties': properties,
            },
        },
    }


emotion_ids = [e['id'] for e in emotions]
speaker_roles = ['elder', 'hunter', 'shaman', 'warrior', 'healer', 'gatherer', 'craftsman', 'chief', 'scout', 'observer']

b_schema = json_schema(
    'task_b',
    ['text_ko', 'text_en', 'register', 'emotion_expressed', 'intensity', 'mimetics', 'temperament_influence'],
    {
        'text_ko': {'type': 'string'},
        'text_en': {'type': 'string'},
        'register': {'type': 'string', 'enum': ['haera']},
        'emotion_expressed': {'type': 'string', 'enum': emotion_ids},
        'intensity': {'type': 'number'},
        'mimetics': {'type': 'array', 'items': {'type': 'string'}},
        'temperament_influence': {'type': 'string'},
    },
)
c_schema = json_schema(
    'task_c',
    ['speech_ko', 'speech_en', 'register', 'emotion_expressed', 'speaker_role', 'temperament_tone'],
    {
        'speech_ko': {'type': 'string'},
        'speech_en': {'type': 'string'},
        'register': {'type': 'string', 'enum': ['haera']},
        'emotion_expressed': {'type': 'string', 'enum': emotion_ids},
        'speaker_role': {'type': 'string', 'enum': speaker_roles},
        'temperament_tone': {'type': 'string'},
    },
)
e_schema_for = lambda option_count: json_schema(
    'task_e',
    ['action_id', 'confidence', 'hint', 'personality_reasoning', 'temperament_factor'],
    {
        'action_id': {'type': 'integer', 'enum': list(range(option_count))},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'hint': {'type': 'string'},
        'personality_reasoning': {'type': 'string', 'enum': ['novelty_seeking', 'harm_avoidance', 'reward_dependence', 'persistence']},
        'temperament_factor': {'type': 'string'},
    },
)
f_schema = json_schema(
    'task_f',
    ['emotion', 'intensity', 'cause', 'previous_emotion', 'transition_type', 'temperament_amplifier'],
    {
        'emotion': {'type': 'string', 'enum': emotion_ids},
        'intensity': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'cause': {'type': 'string'},
        'previous_emotion': {'type': 'string', 'enum': emotion_ids},
        'transition_type': {'type': 'string', 'enum': ['gradual', 'sudden', 'sustained']},
        'temperament_amplifier': {'type': 'string'},
    },
)
g_schema = json_schema(
    'task_g',
    ['interpretation_ko', 'interpretation_en', 'action_tendency', 'confidence', 'register', 'misinterpretation_type', 'temperament_bias'],
    {
        'interpretation_ko': {'type': 'string'},
        'interpretation_en': {'type': 'string'},
        'action_tendency': {'type': 'string', 'enum': ['mobilize', 'defend', 'wait', 'retreat', 'celebrate', 'mourn']},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'register': {'type': 'string', 'enum': ['haera', 'hao', 'hae']},
        'misinterpretation_type': {'type': 'string', 'enum': ['overconfident_literal', 'cautious_reversal', 'optimistic_expansion', 'passive_deferral', 'symbolic_abstraction']},
        'temperament_bias': {'type': 'string'},
    },
)
m_schema = json_schema(
    'task_m',
    ['decision_id', 'confidence', 'dissent_risk', 'reasoning', 'resource_commitment', 'timeline'],
    {
        'decision_id': {'type': 'integer'},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'dissent_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'reasoning': {'type': 'string'},
        'resource_commitment': {'type': 'string', 'enum': ['food', 'tools', 'labor', 'weapons', 'none']},
        'timeline': {'type': 'string', 'enum': ['immediate', 'delayed', 'conditional']},
    },
)
o_schema = json_schema(
    'task_o',
    ['public_claim', 'private_truth', 'deception_style', 'lie_degree', 'detection_risk', 'confidence'],
    {
        'public_claim': {'type': 'string'},
        'private_truth': {'type': 'string'},
        'deception_style': {'type': 'string', 'enum': ['evasion', 'half_truth', 'outright_lie', 'exaggeration', 'omission']},
        'lie_degree': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'detection_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
    },
)
p_schema = json_schema(
    'task_p',
    ['retold_version', 'distortion_type', 'added_detail', 'dropped_detail', 'emotional_charge'],
    {
        'retold_version': {'type': 'string'},
        'distortion_type': {'type': 'string', 'enum': ['exaggeration', 'minimization', 'malicious_twist', 'emotional_coloring', 'detail_invention', 'source_confusion', 'faithful']},
        'added_detail': {'type': 'string'},
        'dropped_detail': {'type': 'string'},
        'emotional_charge': {'type': 'number', 'minimum': -1, 'maximum': 1},
    },
)
q_schema = json_schema(
    'task_q',
    ['trauma_response', 'behavioral_change', 'trigger_situation', 'intensity', 'duration', 'coping_mechanism'],
    {
        'trauma_response': {'type': 'string', 'enum': ['avoidance', 'overprotection', 'aggression', 'withdrawal', 'hypervigilance', 'ritual_coping', 'resilience']},
        'behavioral_change': {'type': 'string'},
        'trigger_situation': {'type': 'string'},
        'intensity': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'duration': {'type': 'string', 'enum': ['short_term', 'long_term', 'permanent']},
        'coping_mechanism': {'type': 'string'},
    },
)
r_schema = json_schema(
    'task_r',
    ['action', 'counter_give', 'counter_want', 'reasoning', 'emotional_state', 'walk_away_threshold'],
    {
        'action': {'type': 'string', 'enum': ['accept', 'counter_offer', 'reject', 'walk_away', 'stall', 'bluff']},
        'counter_give': {'type': 'string'},
        'counter_want': {'type': 'string'},
        'reasoning': {'type': 'string'},
        'emotional_state': {'type': 'string', 'enum': emotion_ids},
        'walk_away_threshold': {'type': 'number', 'minimum': 0, 'maximum': 1},
    },
)
t_schema = json_schema(
    'task_t',
    ['decision_id', 'confidence', 'dissent_risk', 'minority_position', 'minority_action', 'spark_event', 'reasoning', 'timeline'],
    {
        'decision_id': {'type': 'integer'},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'dissent_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'minority_position': {'type': 'integer'},
        'minority_action': {'type': 'string', 'enum': ['comply', 'grumble', 'passive_resist', 'splinter', 'coup_attempt']},
        'spark_event': {'type': 'string', 'enum': ['food_shortage', 'battle_loss', 'oracle_conflict', 'leader_death', 'betrayal', 'resource_discovery']},
        'reasoning': {'type': 'string'},
        'timeline': {'type': 'string', 'enum': ['immediate', 'delayed', 'conditional']},
    },
)

ALL_PROMPTS = []
pair_counter = 0

for sit in pick(situations, 5):
    emotion = rng.choice(emotions)
    intensity = rng.choice(emotion.get('intensities', [0.7]))
    for temp in [TEMPERAMENTS[0], TEMPERAMENTS[1]]:
        pers = rng.choice(personalities)
        pair_counter += 1
        ALL_PROMPTS.append({
            'task': 'B',
            'pair_id': f"B-{pair_counter // 2}",
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"B: {sit['id']} / {temp['id']} / {emotion['id']}",
            'system': SYS_L4,
            'user_prompt': (
                f"[TASK] B\n[TEMP]\n{temp_block(temp)}\n"
                f"[기질 이름] {temp['id']}\n"
                f"[기질 키워드] {temp['keywords']}\n"
                f"[인물 성격] {personality_keywords(pers)}\n"
                f"[지금 느끼는 것] {emotion['id']}:{intensity}\n"
                f"[STRESS] {round(rng.uniform(0.2, 0.9), 1)}\n"
                f"[벌어진 일] {sit.get('desc', sit['id'])}\n"
                '[WORLD] default: 석기시대\n'
                '[출력 형식]\n'
                '{"text_ko": "해라체 2-3문장", "text_en": "English", "register": "haera", "emotion_expressed": "emotion", "intensity": 0.0, "mimetics": [], "temperament_influence": "str"}'
            ),
            'response_format': b_schema,
        })

for sit in pick(situations, 5):
    options = sit.get('action_options', sit.get('typical_actions', []))
    if not options:
        continue
    options_line = ' '.join(f"{i}:{o}" for i, o in enumerate(options))
    emotion = rng.choice(emotions)
    intensity = rng.choice(emotion.get('intensities', [0.7]))
    for temp in [TEMPERAMENTS[0], TEMPERAMENTS[1]]:
        pers = rng.choice(personalities)
        pair_counter += 1
        ALL_PROMPTS.append({
            'task': 'E',
            'pair_id': f"E-{pair_counter // 2}",
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"E: {sit['id']} / {temp['id']}",
            'system': SYS_L3,
            'user_prompt': (
                f"[TASK] E - Action Selection\n[TEMP]\n{temp_block(temp)}\n"
                f"[PERSONALITY] {temp['keywords']}\n"
                f"[EMOTION] {emotion['id']}:{intensity}\n"
                f"[STRESS] {round(rng.uniform(0.2, 0.9), 1)}\n"
                f"[SITUATION] {sit.get('desc', sit['id'])}\n"
                f"[OPTIONS]\n{options_line}\n"
                '[OUTPUT FORMAT]\n'
                '{"action_id": number, "confidence": 0.0-1.0, "hint": "English sentence", "personality_reasoning": "TCI axis", "temperament_factor": "snake_case"}'
            ),
            'response_format': e_schema_for(len(options)),
        })

for sit in pick(situations, 5):
    prev_emotion = rng.choice(emotions)
    for temp in [TEMPERAMENTS[2], TEMPERAMENTS[3]]:
        pers = rng.choice(personalities)
        ALL_PROMPTS.append({
            'task': 'F',
            'pair_id': f"F-{len(ALL_PROMPTS) // 2}",
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"F: {sit['id']} / {temp['id']} (prev={prev_emotion['id']})",
            'system': SYS_L3,
            'user_prompt': (
                f"[TASK] F - Emotion Judgment\n[TEMP]\n{temp_block(temp)}\n"
                f"[PERSONALITY] {temp['keywords']}\n"
                f"[CURRENT EMOTION] {prev_emotion['id']}:{rng.choice(prev_emotion.get('intensities', [0.5]))}\n"
                f"[SITUATION] {sit.get('desc', sit['id'])}\n"
                '[OUTPUT FORMAT]\n'
                f'{{"emotion": "one of 8", "intensity": 0.0-1.0, "cause": "sentence", "previous_emotion": "{prev_emotion["id"]}", "transition_type": "gradual|sudden|sustained", "temperament_amplifier": "str"}}'
            ),
            'response_format': f_schema,
        })

for sit in pick(situations, 5):
    emotion = rng.choice(emotions)
    register = pers_register = 'haera'
    for temp in [TEMPERAMENTS[0], TEMPERAMENTS[1]]:
        pers = rng.choice(personalities)
        ALL_PROMPTS.append({
            'task': 'C',
            'pair_id': None,
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"C: {sit['id']} / {temp['id']} / {emotion['id']}",
            'system': SYS_L4,
            'user_prompt': (
                f"[TASK] C\n[TEMP]\n{temp_block(temp)}\n"
                f"[기질 이름] {temp['id']}\n"
                f"[인물 성격] {personality_keywords(pers)}\n"
                f"[지금 느끼는 것] {emotion['id']}:{rng.choice(emotion.get('intensities', [0.6]))}\n"
                f"[벌어진 일] {sit.get('desc', sit['id'])}\n"
                '[역할] warrior\n'
                '[출력 형식]\n'
                '{"speech_ko": "해라체 대사", "speech_en": "English", "register": "haera", "emotion_expressed": "emotion", "speaker_role": "role", "temperament_tone": "str"}'
            ),
            'response_format': c_schema,
        })

for sit in pick(situations, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'G',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'oracle',
        'scenario_id': sit['id'],
        'desc': f"G: {sit['id']} / {temp['id']}",
        'system': SYS_L5,
        'user_prompt': (
            f"[TASK] G - Oracle Interpretation\n[TEMP]\n{temp_block(temp)}\n"
            f"[기질 이름] {temp['id']}\n"
            f"[인물 성격] {temp['keywords']}\n"
            '[ORACLE]\n"큰물이 밀려오기 전에 높은 곳으로 가라"\n'
            f"[벌어진 일] {sit.get('desc', sit['id'])}\n"
            '[역할] warrior\n'
            '[출력 형식]\n'
            '{"interpretation_ko": "해라체", "interpretation_en": "English", "action_tendency": "enum", "confidence": 0-1, "register": "haera", "misinterpretation_type": "enum", "temperament_bias": "str"}'
        ),
        'response_format': g_schema,
    })

for scenario in pick(deception_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    emotion = rng.choice(emotions)
    ALL_PROMPTS.append({
        'task': 'O',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"O: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] O - Deception\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] {emotion['id']}:{rng.choice(emotion.get('intensities', [0.5]))}\n"
            f"[TRUE_STATE] {scenario['true_state']}\n"
            f"[PUBLIC_GOAL] {scenario['public_goal']}\n"
            f"[TARGET] {scenario['target']}\n"
            f"[DETECTION_CONTEXT] {scenario['detection_context']}\n"
            '[OUTPUT FORMAT]\n'
            '{"public_claim": "str", "private_truth": "str", "deception_style": "enum", "lie_degree": 0-1, "detection_risk": 0-1, "confidence": 0-1}'
        ),
        'response_format': o_schema,
    })

for scenario in pick(rumor_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'P',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"P: {scenario['id']} / {temp['id']} / {scenario['reteller_relationship']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] P - Rumor\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] {rng.choice(emotions)['id']}:{round(rng.uniform(0.3, 0.8), 1)}\n"
            f"[ORIGINAL_FACT] {scenario['original_fact']}\n"
            f"[RETELLER_RELATIONSHIP] {scenario['reteller_relationship']}\n"
            '[OUTPUT FORMAT]\n'
            '{"retold_version": "str", "distortion_type": "enum", "added_detail": "str", "dropped_detail": "str", "emotional_charge": -1 to 1}'
        ),
        'response_format': p_schema,
    })

for scenario in pick(trauma_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'Q',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"Q: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] Q - Trauma\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] sadness:{round(rng.uniform(0.6, 0.95), 2)}\n"
            f"[TRAUMA_EVENT] {scenario['event']}\n"
            f"[TIME_SINCE] {scenario['time_since']}\n"
            f"[CURRENT_SITUATION] {scenario['current_situation']}\n"
            '[OUTPUT FORMAT]\n'
            '{"trauma_response": "enum", "behavioral_change": "str", "trigger_situation": "str", "intensity": 0-1, "duration": "enum", "coping_mechanism": "str"}'
        ),
        'response_format': q_schema,
    })

for scenario in pick(negotiation_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'R',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"R: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] R - Negotiate\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] anticipation:{round(rng.uniform(0.3, 0.7), 1)}\n"
            f"[CONTEXT] {scenario['context']}\n"
            f"[OUR_RESOURCES] {scenario['our_resources']}\n"
            f"[THEIR_RESOURCES] {scenario['their_resources']}\n"
            f"[OFFER_HISTORY] {scenario['offer_history']}\n"
            f"[POWER_BALANCE] {scenario['power_balance']}\n"
            '[OPTIONS]\n0:accept 1:counter_offer 2:reject 3:walk_away 4:stall 5:bluff\n'
            '[OUTPUT FORMAT]\n'
            '{"action": "enum", "counter_give": "str", "counter_want": "str", "reasoning": "str", "emotional_state": "emotion", "walk_away_threshold": 0-1}'
        ),
        'response_format': r_schema,
    })

for scenario in pick(group_situations, 10):
    opts = scenario.get('options', [])
    opts_line = ' '.join(f"{o['id']}:{o.get('ko', o.get('desc', ''))}" for o in opts) if opts else '0:agree 1:disagree 2:delay'
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'M',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"M: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] M - Group Decision\n[TEMP]\nmean_NS=0.5 mean_HA=0.5 var=0.3 leader={temp['id']}\n"
            f"[GROUP_CONTEXT] {scenario.get('group_context', scenario.get('desc', scenario['id']))}\n"
            f"[SITUATION] {scenario.get('situation', scenario.get('desc', scenario['id']))}\n"
            f"[OPTIONS]\n{opts_line}\n"
            '[OUTPUT FORMAT]\n'
            '{"decision_id": number, "confidence": 0-1, "dissent_risk": 0-1, "reasoning": "str", "resource_commitment": "enum", "timeline": "enum"}'
        ),
        'response_format': m_schema,
    })

for scenario in pick(group_dissent_scenarios, 10):
    opts = scenario.get('options', [])
    opts_line = ' '.join(f"{o['id']}:{o['desc']}" for o in opts) if opts else '0:exile 1:trial 2:forgive'
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'T',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"T: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] T - Group Dissent\n[TEMP]\nmean_NS=0.5 mean_HA=0.5 var=0.4 leader={temp['id']}\n"
            f"[GROUP_CONTEXT] {scenario.get('group_context', '')}\n"
            f"[SITUATION] {scenario.get('situation', scenario['id'])}\n"
            f"[FACTION_HINT] {scenario.get('faction_hint', '')}\n"
            f"[OPTIONS]\n{opts_line}\n"
            '[OUTPUT FORMAT]\n'
            '{"decision_id": number, "confidence": 0-1, "dissent_risk": 0-1, "minority_position": number, "minority_action": "enum", "spark_event": "enum", "reasoning": "str", "timeline": "enum"}'
        ),
        'response_format': t_schema,
    })

task_counts = Counter(prompt['task'] for prompt in ALL_PROMPTS)
print('=== Test Prompt Summary ===')
for task, count in sorted(task_counts.items()):
    paired = sum(1 for prompt in ALL_PROMPTS if prompt['task'] == task and prompt.get('pair_id'))
    print(f"  Task {task}: {count} prompts ({paired} paired)")
print(f"  Total: {len(ALL_PROMPTS)} prompts × {len(MODELS)} models = {len(ALL_PROMPTS) * len(MODELS)} inferences")


## 3. Phase 1 — Base Model Test (No Fine-Tuning)

In [ ]:
PHASE1_MODELS = {
    'qwen_08b_ft': {'path': QWEN_08B_FT, 'label': 'Qwen3.5-0.8B FT'},
    'qwen_2b_ft': {'path': QWEN_2B_FT, 'label': 'Qwen3.5-2B FT'},
    'qwen_4b_ft': {'path': QWEN_4B_FT, 'label': 'Qwen3.5-4B FT'},
    'gemma4_e2b_base': {'path': GEMMA4_E2B_IT_GGUF, 'label': 'Gemma4-E2B IT (no FT)'},
    'gemma4_e4b_base': {'path': GEMMA4_E4B_IT_GGUF, 'label': 'Gemma4-E4B IT (no FT)'},
}

PHASE1_RESULTS = []
for model_key, model_info in PHASE1_MODELS.items():
    if not model_info['path'].exists():
        print(f"  Skipping {model_info['label']} - not found")
        continue

    print(f"\n{'=' * 60}")
    print(f"  Testing: {model_info['label']}")
    print(f"{'=' * 60}", flush=True)

    proc = start_server(model_info['path'])
    time.sleep(1)
    try:
        for i, test in enumerate(ALL_PROMPTS):
            messages = [
                {'role': 'system', 'content': test['system']},
                {'role': 'user', 'content': test['user_prompt']},
            ]
            start_t = time.perf_counter()
            output = query_model(messages, response_format=test.get('response_format'), max_tokens=256, temperature=0)
            latency = (time.perf_counter() - start_t) * 1000

            json_valid = False
            parsed = None
            try:
                parsed = json.loads(output)
                json_valid = True
            except Exception:
                pass

            PHASE1_RESULTS.append({
                'model': model_key,
                'model_label': model_info['label'],
                'task': test['task'],
                'desc': test.get('desc', ''),
                'pair_id': test.get('pair_id'),
                'temperament_id': test.get('temperament_id'),
                'output': output,
                'parsed': parsed,
                'json_valid': json_valid,
                'latency_ms': round(latency),
            })
            if (i + 1) % 20 == 0:
                print(f"  [{i + 1}/{len(ALL_PROMPTS)}] model={model_key}", flush=True)
    finally:
        stop_server(proc)
        time.sleep(2)

print(f'\nPhase 1 complete: {len(PHASE1_RESULTS)} results')


## 4. Phase 1 Results — Base Model Comparison

In [ ]:
def auto_grade(result):
    grades = {}
    grades['json_valid'] = result['json_valid']
    placeholder_words = {'str', 'string', 'sentence', 'English', 'enum', 'number', '해라체', 'one of'}

    if result['parsed']:
        values = [str(v) for v in result['parsed'].values() if isinstance(v, str)]
        grades['not_placeholder'] = not any(v in placeholder_words for v in values)
    else:
        grades['not_placeholder'] = False

    if result['parsed']:
        str_fields = [v for v in result['parsed'].values() if isinstance(v, str)]
        avg_len = sum(len(s) for s in str_fields) / max(len(str_fields), 1)
        grades['text_richness'] = avg_len > 15
    else:
        grades['text_richness'] = False

    if result['parsed']:
        grades['numeric_valid'] = True
        for key in ['confidence', 'intensity', 'lie_degree', 'detection_risk', 'dissent_risk', 'walk_away_threshold', 'emotional_charge']:
            if key in result['parsed']:
                val = result['parsed'][key]
                if isinstance(val, (int, float)):
                    if key == 'emotional_charge':
                        grades['numeric_valid'] = -1 <= val <= 1
                    else:
                        grades['numeric_valid'] = 0 <= val <= 1
                else:
                    grades['numeric_valid'] = False
                break
    else:
        grades['numeric_valid'] = False

    grades['score'] = sum([
        grades.get('json_valid', False),
        grades.get('not_placeholder', False),
        grades.get('text_richness', False),
        grades.get('numeric_valid', False),
    ])
    return grades


for row in PHASE1_RESULTS:
    row['grades'] = auto_grade(row)

print('=' * 90)
print('  PHASE 1 - BASE MODEL COMPARISON (Gemma4 IT vs Qwen3.5 FT)')
print('=' * 90)
print(f"\n  {'Model':<30} {'AvgScore':>10} {'JSON%':>7} {'!Placeholder':>13} {'AvgMs':>8} {'GGUF MB':>8}")
print(f"  {'-' * 78}")

for mk in ['qwen_08b_ft', 'qwen_2b_ft', 'qwen_4b_ft', 'gemma4_e2b_base', 'gemma4_e4b_base']:
    mr = [r for r in PHASE1_RESULTS if r['model'] == mk]
    if not mr:
        continue
    n = len(mr)
    avg_score = sum(r['grades']['score'] for r in mr) / n
    json_pct = sum(1 for r in mr if r['grades']['json_valid']) / n * 100
    np_pct = sum(1 for r in mr if r['grades']['not_placeholder']) / n * 100
    avg_ms = sum(r['latency_ms'] for r in mr) / n
    gguf_mb = PHASE1_MODELS[mk]['path'].stat().st_size / 1e6
    label = PHASE1_MODELS[mk]['label']
    print(f"  {label:<30} {avg_score:>8.1f}/4 {json_pct:>6.0f}% {np_pct:>12.0f}% {avg_ms:>7.0f}ms {gguf_mb:>7.0f}")

print('\n  Per Task:')
tasks = sorted(set(r['task'] for r in PHASE1_RESULTS))
for task in tasks:
    line = f'    Task {task}:'
    for mk in ['qwen_2b_ft', 'gemma4_e2b_base', 'gemma4_e4b_base']:
        mr = [r for r in PHASE1_RESULTS if r['task'] == task and r['model'] == mk]
        if mr:
            avg = sum(r['grades']['score'] for r in mr) / len(mr)
            line += f' {mk.split("_")[0][:5]}={avg:.1f}'
    print(line)


## 5. Phase 2 — Fine-Tune Gemma4 E2B + E4B

In [ ]:
import torch
from training.lib.qlora_smoke import SmokeRunConfig, run_baseline_or_raise

train_file = REPO_ROOT / 'data' / 'training' / 'worldsim-v31-mix' / 'train_curriculum.jsonl'
dev_file = REPO_ROOT / 'data' / 'training' / 'worldsim-v31-mix' / 'dev_converted.jsonl'
assert train_file.exists(), f'Training data not found: {train_file}'
assert dev_file.exists(), f'Dev data not found: {dev_file}'

train_rows = sum(1 for _ in open(train_file, encoding='utf-8'))
dev_rows = sum(1 for _ in open(dev_file, encoding='utf-8'))
print(f'Training data: {train_rows} train, {dev_rows} dev')

effective_batch = 1 * 8
max_steps = (train_rows * 3) // effective_batch
print(f'Max steps: {max_steps} (3 epochs)')

GEMMA4_MODELS_TO_TRAIN = [
    {
        'base_model': GEMMA4_E2B_BASE_ID,
        'output_name': 'worldsim-v31-gemma4-e2b',
        'label': 'Gemma4-E2B',
    },
    {
        'base_model': GEMMA4_E4B_BASE_ID,
        'output_name': 'worldsim-v31-gemma4-e4b',
        'label': 'Gemma4-E4B',
    },
]

TRAIN_RESULTS = {}
for model_cfg in GEMMA4_MODELS_TO_TRAIN:
    run_id = datetime.now(UTC).strftime('run-%Y%m%dT%H%M%SZ')
    output_dir = REPO_ROOT / 'outputs' / 'baseline' / model_cfg['output_name'] / run_id

    print(f"\n{'=' * 60}")
    print(f"  Training: {model_cfg['label']}")
    print(f"  Base: {model_cfg['base_model']}")
    print(f"  Output: {output_dir}")
    print(f"{'=' * 60}", flush=True)

    cfg = SmokeRunConfig(
        run_mode='baseline',
        model_name=model_cfg['base_model'],
        train_file=train_file,
        dev_file=dev_file,
        output_dir=output_dir,
        max_steps=max_steps,
        max_train_samples=0,
        max_eval_samples=0,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        logging_steps=10,
        eval_steps=100,
        save_steps=100,
        save_total_limit=1,
        require_qlora=True,
        seed=42,
        dry_run=False,
    )

    started = time.perf_counter()
    train_result = run_baseline_or_raise(cfg)
    elapsed = time.perf_counter() - started

    TRAIN_RESULTS[model_cfg['output_name']] = {
        'train_loss': train_result.train_loss,
        'eval_loss': train_result.eval_loss,
        'elapsed_min': round(elapsed / 60, 1),
        'output_dir': str(train_result.output_dir),
        'base_model': model_cfg['base_model'],
        'label': model_cfg['label'],
    }

    print(f"  train_loss: {train_result.train_loss:.4f}")
    print(f"  eval_loss: {train_result.eval_loss:.4f}")
    print(f"  elapsed: {elapsed / 60:.1f} min")

print(f'\nTraining complete: {len(TRAIN_RESULTS)} models')
for name, info in TRAIN_RESULTS.items():
    print(f"  {info['label']}: train={info['train_loss']:.4f} eval={info['eval_loss']:.4f} ({info['elapsed_min']}min)")


## 6. GGUF Conversion

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

GEMMA4_FT_GGUFS = {}

for model_name, info in TRAIN_RESULTS.items():
    base_model_id = info['base_model']
    adapter_dir = Path(info['output_dir']) / 'adapter'
    merged_dir = Path(info['output_dir']) / 'merged_bf16'
    fp16_gguf = Path(info['output_dir']) / f'{model_name}-f16.gguf'
    q4_gguf = Path(info['output_dir']) / f'{model_name}-q4_k_m.gguf'
    final_gguf = GGUF_DIR / f'{model_name}-q4_k_m.gguf'

    print(f"\n--- Converting {info['label']} ---", flush=True)

    if not (merged_dir / 'config.json').exists():
        print('  Merging LoRA...', flush=True)
        base = AutoModelForCausalLM.from_pretrained(base_model_id, torch_dtype=torch.bfloat16, device_map='cpu')
        tokenizer = AutoTokenizer.from_pretrained(base_model_id)
        merged = PeftModel.from_pretrained(base, str(adapter_dir)).merge_and_unload()
        merged_dir.mkdir(parents=True, exist_ok=True)
        merged.save_pretrained(str(merged_dir))
        tokenizer.save_pretrained(str(merged_dir))
        del merged, base
        torch.cuda.empty_cache()
        import gc
        gc.collect()
        print('  Merged')

    if not fp16_gguf.exists():
        print('  Converting to GGUF FP16...', flush=True)
        r = subprocess.run(
            [sys.executable, str(CONVERT_SCRIPT), str(merged_dir), '--outfile', str(fp16_gguf), '--outtype', 'f16'],
            capture_output=True,
            text=True,
        )
        if r.returncode != 0:
            print(f"  GGUF conversion failed: {r.stderr[-300:]}")
            continue
        print(f"  FP16: {fp16_gguf.stat().st_size / 1e6:.0f} MB")

    if not q4_gguf.exists():
        print('  Quantizing Q4_K_M...', flush=True)
        r = subprocess.run(
            [str(LLAMA_QUANTIZE), str(fp16_gguf), str(q4_gguf), 'Q4_K_M'],
            capture_output=True,
            text=True,
        )
        if r.returncode != 0:
            print(f"  Quantize failed: {r.stderr[-300:]}")
            continue
        print(f"  Q4_K_M: {q4_gguf.stat().st_size / 1e6:.0f} MB")

    shutil.copy2(q4_gguf, final_gguf)
    GEMMA4_FT_GGUFS[model_name] = final_gguf
    print(f"  {final_gguf.name} ({final_gguf.stat().st_size / 1e6:.0f} MB)")

print(f'\nGemma4 FT GGUFs ready: {len(GEMMA4_FT_GGUFS)}')


## 7. Phase 2 — Full 7-Model Comparison

In [ ]:
ALL_MODELS = {
    'qwen_08b_ft': {'path': QWEN_08B_FT, 'label': 'Qwen3.5-0.8B FT'},
    'qwen_2b_ft': {'path': QWEN_2B_FT, 'label': 'Qwen3.5-2B FT'},
    'qwen_4b_ft': {'path': QWEN_4B_FT, 'label': 'Qwen3.5-4B FT'},
    'gemma4_e2b_base': {'path': GEMMA4_E2B_IT_GGUF, 'label': 'Gemma4-E2B IT'},
    'gemma4_e4b_base': {'path': GEMMA4_E4B_IT_GGUF, 'label': 'Gemma4-E4B IT'},
}
if 'worldsim-v31-gemma4-e2b' in GEMMA4_FT_GGUFS:
    ALL_MODELS['gemma4_e2b_ft'] = {'path': GEMMA4_FT_GGUFS['worldsim-v31-gemma4-e2b'], 'label': 'Gemma4-E2B FT'}
if 'worldsim-v31-gemma4-e4b' in GEMMA4_FT_GGUFS:
    ALL_MODELS['gemma4_e4b_ft'] = {'path': GEMMA4_FT_GGUFS['worldsim-v31-gemma4-e4b'], 'label': 'Gemma4-E4B FT'}

FINAL_RESULTS = []
for model_key, model_info in ALL_MODELS.items():
    if not model_info['path'].exists():
        print(f"  Skipping {model_info['label']}")
        continue
    print(f"\n{'=' * 60}")
    print(f"  Testing: {model_info['label']}")
    print(f"{'=' * 60}", flush=True)

    proc = start_server(model_info['path'])
    time.sleep(1)
    try:
        for i, test in enumerate(ALL_PROMPTS):
            messages = [
                {'role': 'system', 'content': test['system']},
                {'role': 'user', 'content': test['user_prompt']},
            ]
            start_t = time.perf_counter()
            output = query_model(messages, response_format=test.get('response_format'), max_tokens=256, temperature=0)
            latency = (time.perf_counter() - start_t) * 1000
            json_valid = False
            parsed = None
            try:
                parsed = json.loads(output)
                json_valid = True
            except Exception:
                pass
            FINAL_RESULTS.append({
                'model': model_key,
                'model_label': model_info['label'],
                'task': test['task'],
                'desc': test.get('desc', ''),
                'pair_id': test.get('pair_id'),
                'temperament_id': test.get('temperament_id'),
                'output': output,
                'parsed': parsed,
                'json_valid': json_valid,
                'latency_ms': round(latency),
            })
            if (i + 1) % 20 == 0:
                print(f"  [{i + 1}/{len(ALL_PROMPTS)}] model={model_key}", flush=True)
    finally:
        stop_server(proc)
        time.sleep(2)

for row in FINAL_RESULTS:
    row['grades'] = auto_grade(row)

print(f'\nPhase 2 complete: {len(FINAL_RESULTS)} results')


## 8. Full Comparison Table

In [ ]:
print('=' * 100)
print('  GEMMA 4 vs QWEN 3.5 - FULL COMPARISON')
print('=' * 100)

print(f"\n  {'Model':<25} {'Score':>7} {'JSON%':>7} {'!Plchld':>8} {'AvgMs':>8} {'GGUF':>8} {'Family':>8}")
print(f"  {'-' * 72}")

model_order = ['qwen_08b_ft', 'qwen_2b_ft', 'qwen_4b_ft', 'gemma4_e2b_base', 'gemma4_e2b_ft', 'gemma4_e4b_base', 'gemma4_e4b_ft']
for mk in model_order:
    mr = [r for r in FINAL_RESULTS if r['model'] == mk]
    if not mr:
        continue
    n = len(mr)
    avg_score = sum(r['grades']['score'] for r in mr) / n
    json_pct = sum(1 for r in mr if r['grades']['json_valid']) / n * 100
    np_pct = sum(1 for r in mr if r['grades']['not_placeholder']) / n * 100
    avg_ms = sum(r['latency_ms'] for r in mr) / n
    gguf_mb = ALL_MODELS[mk]['path'].stat().st_size / 1e6
    family = 'Qwen' if 'qwen' in mk else 'Gemma4'
    ft = 'FT' if 'ft' in mk else 'base'
    print(f"  {ALL_MODELS[mk]['label']:<25} {avg_score:>5.1f}/4 {json_pct:>6.0f}% {np_pct:>7.0f}% {avg_ms:>7.0f}ms {gguf_mb:>6.0f}MB {family:>6}:{ft}")

print('\n  Per Task (Score/4):')
print(f"  {'Task':<8}", end='')
for mk in model_order:
    mr = [r for r in FINAL_RESULTS if r['model'] == mk]
    if mr:
        short = mk.replace('qwen_', 'Q').replace('gemma4_', 'G4').replace('_ft', 'F').replace('_base', 'B')
        print(f" {short:>7}", end='')
print()
print(f"  {'-' * 8}" + '-' * 8 * sum(1 for mk in model_order if any(r['model'] == mk for r in FINAL_RESULTS)))

for task in sorted(set(r['task'] for r in FINAL_RESULTS)):
    print(f"  {task:<8}", end='')
    for mk in model_order:
        mr = [r for r in FINAL_RESULTS if r['model'] == mk and r['task'] == task]
        if mr:
            avg = sum(r['grades']['score'] for r in mr) / len(mr)
            print(f" {avg:>6.1f}", end=' ')
    print()


## 9. Personality Consistency

In [ ]:
print('=' * 80)
print('  PERSONALITY CONSISTENCY - ALL MODELS')
print('=' * 80)

pairs = defaultdict(list)
for row in FINAL_RESULTS:
    pair_id = row.get('pair_id')
    if pair_id:
        pairs[(pair_id, row['model'])].append(row)

consistency = defaultdict(lambda: {'same': 0, 'different': 0, 'total': 0})
for (pair_id, model_key), pair_rows in sorted(pairs.items()):
    if len(pair_rows) != 2:
        continue
    first, second = pair_rows
    if not first['parsed'] or not second['parsed']:
        continue
    task = first['task']
    different = False
    if task == 'E':
        different = first['parsed'].get('action_id') != second['parsed'].get('action_id')
    elif task == 'B':
        different = first['parsed'].get('emotion_expressed') != second['parsed'].get('emotion_expressed') or abs(first['parsed'].get('intensity', 0) - second['parsed'].get('intensity', 0)) > 0.1
    elif task == 'F':
        different = first['parsed'].get('emotion') != second['parsed'].get('emotion')
    consistency[model_key]['total'] += 1
    if different:
        consistency[model_key]['different'] += 1
    else:
        consistency[model_key]['same'] += 1

print(f"\n  {'Model':<25} {'Pairs':>6} {'Diff':>6} {'Same':>6} {'Consistency%':>13}")
print(f"  {'-' * 58}")
for mk in model_order:
    stats = consistency.get(mk, {'total': 0, 'different': 0, 'same': 0})
    if stats['total'] > 0:
        pct = stats['different'] / stats['total'] * 100
        print(f"  {ALL_MODELS[mk]['label']:<25} {stats['total']:>6} {stats['different']:>6} {stats['same']:>6} {pct:>12.0f}%")


## 10. WorldSim Recommendation

In [ ]:
print('=' * 80)
print('  WORLDSIM MODEL RECOMMENDATION')
print('=' * 80)

for qwen_key, gemma_key, tier in [
    ('qwen_2b_ft', 'gemma4_e2b_ft', 'Tier 2 (2B class)'),
    ('qwen_4b_ft', 'gemma4_e4b_ft', 'Tier 3 (4B class)'),
]:
    qr = [r for r in FINAL_RESULTS if r['model'] == qwen_key]
    gr = [r for r in FINAL_RESULTS if r['model'] == gemma_key]
    if not qr or not gr:
        continue

    q_score = sum(r['grades']['score'] for r in qr) / len(qr)
    g_score = sum(r['grades']['score'] for r in gr) / len(gr)
    q_json = sum(1 for r in qr if r['grades']['json_valid']) / len(qr) * 100
    g_json = sum(1 for r in gr if r['grades']['json_valid']) / len(gr) * 100
    q_ms = sum(r['latency_ms'] for r in qr) / len(qr)
    g_ms = sum(r['latency_ms'] for r in gr) / len(gr)
    q_mb = ALL_MODELS[qwen_key]['path'].stat().st_size / 1e6
    g_mb = ALL_MODELS[gemma_key]['path'].stat().st_size / 1e6

    q_cons = consistency.get(qwen_key, {'different': 0, 'total': 1})
    g_cons = consistency.get(gemma_key, {'different': 0, 'total': 1})
    q_pct = q_cons['different'] / max(q_cons['total'], 1) * 100
    g_pct = g_cons['different'] / max(g_cons['total'], 1) * 100

    print(f"\n  {tier}:")
    print(f"    {'':15} {'Score':>7} {'JSON':>6} {'Ms':>7} {'GGUF':>7} {'Pers%':>7}")
    print(f"    {ALL_MODELS[qwen_key]['label']:<15} {q_score:>5.1f}/4 {q_json:>5.0f}% {q_ms:>6.0f}ms {q_mb:>5.0f}MB {q_pct:>6.0f}%")
    print(f"    {ALL_MODELS[gemma_key]['label']:<15} {g_score:>5.1f}/4 {g_json:>5.0f}% {g_ms:>6.0f}ms {g_mb:>5.0f}MB {g_pct:>6.0f}%")

    score_win = 'Gemma4' if g_score > q_score + 0.1 else 'Qwen' if q_score > g_score + 0.1 else 'Tie'
    size_win = 'Qwen' if q_mb < g_mb * 0.8 else 'Gemma4' if g_mb < q_mb * 0.8 else 'Similar'
    speed_win = 'Qwen' if q_ms < g_ms * 0.9 else 'Gemma4' if g_ms < q_ms * 0.9 else 'Similar'

    print(f'    -> Quality: {score_win} | Size: {size_win} | Speed: {speed_win}')

print('\n  Key question: Does Gemma4 quality justify the larger GGUF footprint?')
print('  For 8GB PC deployment, Qwen3.5 0.8B+2B remains the smaller deployment baseline.')


## 11. Save Results

In [ ]:
results_path = REPO_ROOT / 'outputs' / 'benchmarks' / 'gemma4_full_benchmark.json'
results_path.parent.mkdir(parents=True, exist_ok=True)

save_data = {
    'metadata': {
        'gemma4_models': ['E2B', 'E4B'],
        'qwen_models': ['0.8B', '2B', '4B'],
        'training_data': 'worldsim-v31-mix',
        'train_rows': train_rows,
        'total_results': len(FINAL_RESULTS),
        'training_results': TRAIN_RESULTS,
    },
    'results': [{k: v for k, v in row.items() if k != 'parsed'} for row in FINAL_RESULTS],
}

with open(results_path, 'w', encoding='utf-8') as handle:
    json.dump(save_data, handle, indent=2, ensure_ascii=False)

print(f'Saved: {results_path}')
